<div align="right"><a href="L33-Yule-Walker-und-Autokorrelationsmethode.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/S33-Yule-Walker-und-Autokorrelationsmethode.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S32 Power Spectral Density Estimation using the Bartlett-Welch Method](./S32-PSD-Estimation-using-the-Bartlett-Welch-Method.ipynb) | [Exercise 33](./E33-Yule-Walker-und-Autokorrelationsmethode.ipynb) | [Contents / Overview](./00-Python-Notebooks-Overview-EN.ipynb) | Next: [S34 Burg-Algorithm](./S34-Burg-Algorithm.ipynb)

</div>

---


### Exercise: Yule–Walker

The goal of this exercise is to analyze spectrum estimation by applying the Yule–Walker resp. the autocorrelation approach. It is crucial for this procedure that the process $X(k)$, which is to be analyzed, is set to zero outside the measured interval $0 \leq k \leq N-1$. Thus, the *transient behavior* of the predictor is taken into account when calculating the prediction error (eq. (9.3.7)). In spite of the influence of the transient behavior, this approach is denoted as stationary. The reason for this is the calculation of the coefficients of the correlation matrix, which has the form of a Hermitian Toeplitz matrix. Such a matrix also occurs as an autocorrelation matrix of stationary processes.

In addition, the biased autocorrelation series $\hat{r}_{XX}(\kappa)$ is used. Therefore, the estimation of the AR coefficients is biased as well.

For the analysis, an (all-pole) system $H(z)=1/A(z)$ is used with

$$
A(z)=1 + 0.3611z^{-1} - 0.1373z^{-2} + 0.2685z^{-3} - 0.0533z^{-4} - 0.3353z^{-5} - 0.307z^{-6}
$$

of order $m=6$, which is excited by a stationary, zero-mean, white process $N(k)$ with power $\sigma_N^2 = 1$.

<img src="./pngs/l15sys.png" width="70%">

---

#### a)

Plot the impulse response $h(k)$ of the system within the interval $0 \leq k \leq 80$ using an appropriate Python function (e.g. `scipy.signal.impulse` or by filtering a unit impulse), and determine the pole–zero plane (e.g. using `scipy.signal.tf2zpk`).

How many samples are necessary for the system to tune in, i.e. such that $h(k) \approx 0$?

---

#### b)

Determine the exact autocorrelation series $r_{XX}(\kappa)$ using the impulse response for $0 \leq \kappa \leq 6$. Calculate the exact coefficients $a_0,\ldots,a_n$ of the AR($n$) model using the Yule–Walker equation (9.3.17):

$$
\frac{1}{A_n(z)} = \frac{1}{a_0 + a_1 z^{-1} + \cdots + a_n z^{-n}},
\qquad \text{with} \quad n=6
$$

(For this special case, $n$ is set to $6$, so that the order of the AR model equals the order of the system $H(z)$.)

Furthermore, calculate the exact power density spectrum $S_{XX}(\mathrm{e}^{j\Omega})$ and plot the results.

---

#### c)

Next, analyze the influence of the sample length $N$ of the stochastic process $X(k)$ on the estimation of the power density spectrum. For this purpose, generate $1000$ realizations of the process $X(k)$. It must be ensured that the system has tuned in within the interval of $N$ samples, i.e. $h(k) \approx 0$ (e.g. by discarding an initial transient when using `scipy.signal.lfilter`).

The measurement interval should include $N=10$ and $N=256$ samples, respectively. The calculations should be performed using a Python implementation of the Yule–Walker method with biased autocorrelation.

First, determine the estimated coefficients $\hat{a}*{6,1},\ldots,\hat{a}*{6,6}$ of the AR(6) model, as well as the mean value and the variance of each coefficient using all $1000$ ensembles. Plot the results.

To analyze the bias, determine the power of the deviation between the mean values and the exact AR coefficients:

$$
\frac{1}{n}\sum_{i=0}^{n} \left| \bar{a}_i - a_i \right|^2
$$

where $\bar{a}_i$ denotes the mean value and $a_i$ the exact coefficient.

Based on the estimated coefficients, calculate the power density spectrum for each ensemble. Determine the mean power density spectrum as well as the variance of the estimation. Plot the exact spectrum, the mean estimated spectrum, and the upper $2\sigma$ bounds (95% bounds) together in a single figure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import lyw, zplane
# BEGIN SOLUTION
MA = np.array([1.0])
AR = np.array([1.0000, 0.3611, -0.1373, 0.2685, -0.0533, -0.3353, -0.307])
NFFT = 2**10
n = len(AR) - 1

# Task a: Pole-Zero Plot & Impulse Response  
# Execute your zplane function
zplane(MA, AR)

# Impulse response
impulse = np.zeros(80)
impulse[0] = 1.0
h_arma = signal.lfilter(MA, AR, impulse)

plt.figure()
plt.stem(np.arange(1, len(h_arma)+1), h_arma)
plt.title(f'Impulse Response of the AR({n}) Model')
plt.grid(True)
plt.show()
print('Samples required for transient settling: N_vor=80')

# --- Task b: Exact ACF, YW-Coefficients, and PSD ---
# Exact ACF via h_arma
rxx = signal.correlate(h_arma, h_arma, mode='full')
center = len(rxx) // 2
rxx_ex = rxx[center : center + n + 1]

ARcoeff_b, _, _ = lyw(h_arma, n, 0)
Omega_norm = np.linspace(0, 1, NFFT, endpoint=False)

# Frequency response calculation (PSD)
w, h_freq = signal.freqz(MA, AR, worN=NFFT, whole=True)
Sxx_ex = np.abs(h_freq)**2

plt.figure()
plt.stem(np.arange(n+1), rxx_ex, linefmt='r-', markerfmt='ro')
plt.grid(True)
plt.xlabel(r'$\kappa$')
plt.ylabel(r'$r_{xx}(\kappa)$')
plt.title(f'Exact ACF values for AR({n}): $r_{{xx}}(0)...r_{{xx}}({n})$')

plt.figure()
plt.stem(np.arange(n+1), ARcoeff_b, linefmt='r-', markerfmt='ro')
plt.grid(True)
plt.ylim([1.1*np.min(ARcoeff_b), 1.1*np.max(ARcoeff_b)])
plt.xlabel('i')
plt.ylabel('$a_i$')
plt.title(f'Exact AR({n}) Coefficients: $a_0...a_{{{n}}}$')

plt.figure()
plt.plot(Omega_norm, 10 * np.log10(Sxx_ex / np.max(Sxx_ex)), 'r--')
plt.grid(True)
plt.xlabel(r'$\Omega/2\pi$')
plt.ylabel(r'$S_{xx}(e^{j\Omega})$ in dB')
plt.title('Exact PSD')

# --- Task c: Influence of Data Length N (Monte Carlo) ---
N_list = [10, 256]
N_vor = 80
mc_run = 1000

print('Calculating Monte Carlo simulation...')

for N_val in N_list:
    Sxx_ar_stack = np.zeros((mc_run, NFFT))
    Coeff_stack = np.zeros((mc_run, n + 1))
    
    for r in range(mc_run):
        noise = np.random.randn(N_val + N_vor)
        x_all = signal.lfilter(MA, AR, noise)
        x = x_all[N_vor:] # Remove transient effect
        
        coeffs, _, _ = lyw(x, n, 1)
        Coeff_stack[r, :] = coeffs
        
        # Estimate PSD from coefficients
        _, h_est = signal.freqz([1.0], coeffs, worN=NFFT, whole=True)
        Sxx_ar_stack[r, :] = np.abs(h_est)**2
        
    Coeff_mean = np.mean(Coeff_stack, axis=0)
    Coeff_std = np.std(Coeff_stack, axis=0)
    C2sig_o = Coeff_mean + 2*Coeff_std
    C2sig_u = Coeff_mean - 2*Coeff_std
    Coeff_mse = np.mean(np.abs(AR - Coeff_mean)**2)
    
    Sxx_mean = np.mean(Sxx_ar_stack, axis=0)
    Sxx_std = np.std(Sxx_ar_stack, axis=0)
    Sxx_2sigma = Sxx_mean + 2*Sxx_std

    # Plot AR Coefficients Statistics
    plt.figure()
    plt.plot(np.arange(n+1), AR, 'ro', label='Exact', mfc='none')
    plt.plot(np.arange(n+1), Coeff_mean, 'bx', label=f'Mean / MSE={Coeff_mse:.2e}')
    plt.plot(np.arange(n+1), C2sig_o, 'go', label=f'2$\sigma$ / $\sigma_{{avg}}$={np.mean(Coeff_std):.2f}')
    plt.plot(np.arange(n+1), C2sig_u, 'go')
    
    # Draw vertical lines for the confidence interval
    for i in range(n+1):
        plt.plot([i, i], [C2sig_u[i], C2sig_o[i]], 'g')
        
    plt.title(f'AR({n}) Coefficients; N={N_val}')
    plt.xlabel('i')
    plt.ylabel('Coefficients')
    plt.legend()
    plt.grid(True)
    plt.ylim([-1.5, 1.5])

    # Plot PSD Statistics
    plt.figure()
    plt.plot(Omega_norm, 10*np.log10(Sxx_ex/np.max(Sxx_ex)), 'r--', label='Exact')
    plt.plot(Omega_norm, 10*np.log10(Sxx_mean/np.max(Sxx_ex)), 'b-', label='Mean')
    plt.plot(Omega_norm, 10*np.log10(Sxx_2sigma/np.max(Sxx_ex)), 'g-.', label='2$\sigma$ Bound')
    plt.ylabel(r'$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel(r'$\Omega/2\pi$')
    plt.title(f'PSD of YW-AR({n}); N={N_val}')
    plt.legend()
    plt.grid(True)

plt.show()
# END SOLUTION

### Solution: Yule–Walker

* **a)**
  The following figures show the impulse response $h(k)$ of the AR(6) system in the interval
  $0 \leq k \leq 80$. It can be stated that the underlying system has reached its steady state
  for time indices $k \geq 60$.

  Additionally, the positions of the poles
  $z_{\infty,1}=0.3444 + i\cdot 0.8315$,
  $z_{\infty,2}=z_{\infty,1}^{*}$,
  $z_{\infty,3}=0.85$,
  $z_{\infty,4}=-0.91$,
  $z_{\infty,5}=-0.49 + i\cdot 0.4950$,
  $z_{\infty,6}=z_{\infty,5}^{*}$
  are depicted in the pole–zero diagram.

  <img src="./pngs/ldsv9a02_a1.png" width="45%">
  <img src="./pngs/ldsv9a02_a2.png" width="45%">

---

* **b)**
  The relation
  $r_{NN}(\kappa)=\sigma_N^2\delta(\kappa) ;\Leftrightarrow;
  S_{NN}(\mathrm{e}^{j\Omega})=\sigma_N^2$
  applies for stationary, zero-mean, white noise. At the system output, the true
  autocorrelation function (ACF) and the exact spectral power density exist with
  $\sigma_N^2=1$:

  $$
  r_{XX}(\kappa)
  = r_{HH}^{(E)}(\kappa) \ast r_{NN}(\kappa)
  = \sigma_N^2 \cdot r_{hh}^{(E)}(\kappa)
  = r_{HH}^{(E)}(\kappa)
  := \sum_{k=-\infty}^{\infty} h^*(k),h(k+\kappa)
  $$

  $$
  S_{XX}(\mathrm{e}^{j\Omega})
  = |H(\mathrm{e}^{j\Omega})|^2 \cdot S_{NN}(\mathrm{e}^{j\Omega})
  = \sigma_N^2 \cdot |H(\mathrm{e}^{j\Omega})|^2
  = |H(\mathrm{e}^{j\Omega})|^2
  $$

  For the calculation of the AR coefficients using the Yule–Walker equation (9.3.17),
  the autocorrelation series is considered at the positions
  $r_{XX}(0), \ldots, r_{XX}(n)$. The parameter $n$ describes the order of the AR($n$)
  process and does not necessarily have to match the order $m$ of the underlying
  system $H(z)$ (underestimation or overestimation of the number of poles).

  In addition to the exact ACF series, the exact AR(6) coefficients
  $a_0,\ldots,a_6$ are shown in the following figure. The computation is performed
  using the Levinson–Durbin recursion to solve the Yule–Walker equation (9.3.17).

  For the power density spectra (PDS)
  $S_{XX}(\mathrm{e}^{j\Omega}) \leftrightarrow r_{XX}(\kappa)$, the trend is plotted
  as a dashed line in the lower figure. The power density spectra are computed using

  $$
  S_{XX}(\mathrm{e}^{j\Omega})
  = \sigma_N^2 \cdot |H(\mathrm{e}^{j\Omega})|^2
  = \left| \frac{B(\mathrm{e}^{j\Omega})}{A(\mathrm{e}^{j\Omega})} \right|^2
  = \frac{\left| \mathcal{F}{b(k)} \right|^2}{\left| \mathcal{F}{a(k)} \right|^2}.
  $$

  For a graphical representation of the true power density spectrum
  $S_{XX}(\mathrm{e}^{j\Omega})$, the discrete-time Fourier transform
  $\mathcal{F}{\cdot}$ is approximated by a $2^{10}$-point FFT.
  Since the transmission system $H(z)$ is real-valued, a symmetric power density
  spectrum normalized by the frequency $\Omega/2\pi$ is obtained.

  <img src="./pngs/ldsv9a02_b1.png" width="30%">
  <img src="./pngs/ldsv9a02_b2.png" width="30%">
  <img src="./pngs/ldsv9a02_b3.png" width="30%">

---

* **c)**
  The left-hand figure shows the results of the coefficient estimation of an AR(6) model
  obtained from $1000$ realizations of the output process $X(k)$ with different sample
  lengths $N=10$ and $N=256$. It is assumed that the system $H(z)$ has reached its steady
  state, i.e. transient samples are not included in the data records.

  For $N=10$, the bias of the autocorrelation method is clearly visible: the mean values
  (crosses) deviate strongly from the exact coefficients (circles). This behavior is
  characterized by the mean squared error $E{e^2}$, which serves as a measure of the
  bias and takes a non-zero value in this case. Furthermore, a relatively large estimation
  variance occurs for the small sample size $N=10$, as indicated by the standard deviation
  $\sigma$.

  A comparison of the coefficient estimates for $N=10$ and $N=256$ shows a clear
  improvement with increasing $N$. The deviation between the mean and the exact value
  is significantly smaller for $N=256$, but still not zero, as illustrated for example
  by $\hat{a}_{6,6}$. The estimation variance and standard deviation also decrease.

  The right-hand figure compares the exact power density spectrum with the mean estimated
  power density spectrum. To visualize the estimation variance, the upper $2\sigma$ bounds
  are also shown. The results clearly demonstrate the influence of the observation interval
  length on the estimation quality. For the considered system $H(z)$, a relatively good
  approximation of the exact power density spectrum is achieved for $N=256$ using the
  autocorrelation method. Nevertheless, it must be emphasized that the estimation is
  inherently biased.

  <img src="./pngs/ldsv9a02_c1.png" width="22%">
  <img src="./pngs/ldsv9a02_c2.png" width="22%">
  <img src="./pngs/ldsv9a02_c3.png" width="22%">
  <img src="./pngs/ldsv9a02_c4.png" width="22%">